# Task 6: Reproducible Monitoring Baselines

This notebook contains the evidence calculated from the implemented system and its saved outputs.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS = PROJECT_ROOT / "outputs"

from src.config import Config
from src.data_loader import load_training_data
from src.preprocessing import preprocess
from src.train import split_data

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\josep\Desktop\CA_Code_Starter\CA_Code_Starter


## 1. Preprocessing quality baseline

This calculation establishes the normal rate of messages that contain no interaction content after cleaning. A material production increase would indicate input-format drift or preprocessing problems.

In [2]:
raw_df = load_training_data()
cleaned_df = preprocess(raw_df)

content_empty = cleaned_df[Config.INTERACTION_CONTENT].str.strip().eq("")
summary_empty = cleaned_df[Config.TICKET_SUMMARY].str.strip().eq("")
both_empty = content_empty & summary_empty

token_count = (
    cleaned_df[Config.TEXT_COLS]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.split()
    .str.len()
)

quality_baseline = pd.DataFrame([
    {"Measure": "Training messages", "Value": len(cleaned_df)},
    {"Measure": "Empty interaction content after cleaning", "Value": int(content_empty.sum())},
    {"Measure": "Empty interaction-content rate", "Value": float(content_empty.mean())},
    {"Measure": "Both text fields empty after cleaning", "Value": int(both_empty.sum())},
    {"Measure": "Mean cleaned token count", "Value": float(token_count.mean())},
    {"Measure": "Median cleaned token count", "Value": float(token_count.median())},
])

display(quality_baseline.style.format({"Value": lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)}))

,Measure,Value
0,Training messages,206.000
1,Empty interaction content after cleaning,8.000
2,Empty interaction-content rate,0.039
3,Both text fields empty after cleaning,0.000
4,Mean cleaned token count,208.874
5,Median cleaned token count,147.500


## 2. Training-label distribution baseline

Production prediction distributions can be compared with these training proportions as a lightweight drift signal. A shift is an alert for investigation rather than proof of performance degradation.

In [3]:
distribution_rows = []
for label in Config.CLASS_COLS:
    counts = cleaned_df[label].value_counts(dropna=False)
    for class_name, count in counts.items():
        distribution_rows.append({
            "Label": label,
            "Class": class_name,
            "Training count": int(count),
            "Training proportion": float(count / len(cleaned_df)),
        })

label_distribution = pd.DataFrame(distribution_rows)
display(label_distribution.style.format({"Training proportion": "{:.1%}"}))

,Label,Class,Training count,Training proportion
0,y2,Suggestion,92,44.7%
1,y2,Problem/Fault,79,38.3%
2,y2,Others,35,17.0%
3,y3,Payment,57,27.7%
4,y3,unknown,35,17.0%
5,y3,Coupon/Gifts/Points Issues,26,12.6%
6,y3,AppGallery-Install/Upgrade,19,9.2%
7,y3,VIP / Offers / Promotions,13,6.3%
8,y3,Third Party APPs,11,5.3%
9,y3,General,10,4.9%


## 3. Classes unseen during training

The reproducible hold-out split is checked for classes that occur in the test set but not in training. the classes can't be learned by either supervised model and should be prioritised when collecting human feedback.

In [4]:
train_df, test_df = split_data(cleaned_df)

unseen_rows = []
for label in Config.CLASS_COLS:
    unseen = sorted(set(test_df[label]) - set(train_df[label]))
    if unseen:
        for class_name in unseen:
            unseen_rows.append({
                "Label": label,
                "Class absent from training": class_name,
                "Test occurrences": int((test_df[label] == class_name).sum()),
            })

unseen_classes = pd.DataFrame(unseen_rows)
display(unseen_classes if not unseen_classes.empty else pd.DataFrame({"Result": ["No unseen test classes"]}))

y3: 1 test class(es) never seen in training: ['Refund']
y4: 1 test class(es) never seen in training: ['Within 14 days of purchase (not product issue)']


,Label,Class absent from training,Test occurrences
0,y3,Refund,1
1,y4,Within 14 days of purchase (not product issue),1


## 4. Deployed-model performance baseline

Periodic human-labelled samples should be evaluated against these saved Random Forest results. Proxy signals such as confidence, unknown rate, and vocabulary drift provide early warnings, but labelled macro F1 and exact-match accuracy are the authoritative degradation measures.

In [5]:
with open(OUTPUTS / "metrics_random_forest.json", encoding="utf-8") as file:
    rf_metrics = json.load(file)

overall = rf_metrics["overall"]
performance_baseline = pd.DataFrame([
    {"Scope": "Overall", "Metric": "Mean macro F1", "Baseline": overall["mean_f1_macro"], "Preferred direction": "Higher"},
    {"Scope": "Overall", "Metric": "Exact-match accuracy", "Baseline": overall["exact_match_accuracy"], "Preferred direction": "Higher"},
    {"Scope": "Overall", "Metric": "Hamming loss", "Baseline": overall["hamming_loss"], "Preferred direction": "Lower"},
])

for label, metrics in rf_metrics["per_label"].items():
    performance_baseline.loc[len(performance_baseline)] = {
        "Scope": label,
        "Metric": "Macro F1",
        "Baseline": metrics["f1_macro"],
        "Preferred direction": "Higher",
    }

print(f"Model: {rf_metrics['model_name']} (version {rf_metrics['model_version']})")
print(f"Test samples: {rf_metrics['n_test_samples']}")
display(performance_baseline.style.format({"Baseline": "{:.3f}"}))

Model: random_forest (version 0.1.0)
Test samples: 42


,Scope,Metric,Baseline,Preferred direction
0,Overall,Mean macro F1,0.525,Higher
1,Overall,Exact-match accuracy,0.524,Higher
2,Overall,Hamming loss,0.325,Lower
3,y2,Macro F1,0.662,Higher
4,y3,Macro F1,0.482,Higher
5,y4,Macro F1,0.431,Higher
